# Ascend C 自定义算子开发循序渐进教程

> **运行环境**：cann_9.0.0-py3.11-A2-arm-20260715 | ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB
## 教学说明

本教程旨在通过**四个递进式实验**，引导学生逐步理解 Ascend C 算子开发的核心概念与编程范式。四个实验分别对应算子开发的四个关键维度：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">阶段</th>
<th style="text-align: left;">实验内容</th>
<th style="text-align: left;">教学重点</th>
<th style="text-align: left;">核心变化</th>
</tr>
<tr>
<td style="text-align: left;"><strong>阶段一</strong></td>
<td style="text-align: left;">双向量加法（8核）</td>
<td style="text-align: left;">掌握算子开发基本流程</td>
<td style="text-align: left;">基准实现</td>
</tr>
<tr>
<td style="text-align: left;"><strong>阶段二</strong></td>
<td style="text-align: left;">双向量加法（32核）</td>
<td style="text-align: left;">理解多核并行与数据切分</td>
<td style="text-align: left;">核数 8→32</td>
</tr>
<tr>
<td style="text-align: left;"><strong>阶段三</strong></td>
<td style="text-align: left;">三向量加法</td>
<td style="text-align: left;">掌握多输入算子的实现</td>
<td style="text-align: left;">输入 2→3</td>
</tr>
<tr>
<td style="text-align: left;"><strong>阶段四</strong></td>
<td style="text-align: left;">双向量乘法</td>
<td style="text-align: left;">理解不同计算指令的使用</td>
<td style="text-align: left;">Add→Mul</td>
</tr>
</table>

**四阶段设计理念详解**：
- **阶段一（基准实现）**：以最简单的双向量加法为切入点，使用 8 个核处理 (8, 2048) 的数据。学生在此阶段掌握 Ascend C 算子开发的完整流程：头文件引入→Tiling 结构体→算子类（Init/Process/CopyIn/Compute/CopyOut）→核函数定义→Host 侧调用→编译运行→结果验证。
- **阶段二（多核扩展）**：保持算子逻辑不变，将核数从 8 增加到 32，数据总量相应增至 (32, 2048)。学生理解 `blockDim` 与 `GetBlockNum()` 的关系，观察核数增加对性能的影响。
- **阶段三（多输入扩展）**：从两个输入向量扩展到三个，计算 `z = x + y + w`。学生需要增加队列、GlobalTensor、内存管理等，理解多输入算子的资源管理。
- **阶段四（指令变化）**：将加法指令 `Add` 替换为乘法指令 `Mul`，计算 `z = x * y`。学生理解不同矢量计算指令的使用方法，体会算子移植的最小修改原则。

> **循序渐进的设计**：每个阶段只引入一个变化点，避免学生同时面对多个新概念。通过对比相邻阶段的代码差异，可以清晰地看到"改了什么、为什么改"，降低学习坡度。

### 教学建议

- **阶段一**：教师讲解+学生跟随操作，建立整体认知
- **阶段二**：学生自主修改，观察核数变化对性能的影响
- **阶段三**：引导学生思考多输入场景下的资源管理
- **阶段四**：学生独立完成，验证对算子开发流程的掌握程度

> 本教程基于 Kernel 直调工程，算子实现与调用代码在同一源文件中，可快速完成算子的开发与调用测试。

---


## 环境准备

正式开始学习之前，先对 Jupyter 环境进行初始化。以下代码完成了初始化并将环境变量导入，同时创建了代码目录 `Sources`。

In [ ]:
!mkdir -p Sources

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 环境初始化完成！")


---

# 阶段一：双向量加法（8核）—— 基准实现

## 学习目标

- 了解 Ascend C 算子开发的完整流程
- 理解核函数、算子类、三级流水（CopyIn-Compute-CopyOut）的基本概念
- 掌握 Kernel 直调工程的开发模式

## 算子分析

本阶段基于 Add 算子为例进行自定义算子开发，特点如下：

- 输入 shape 固定为 (8, 2048)
- 输入类型仅支持 float
- 固定使用 8 个核

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">规格</th>
</tr>
<tr>
<td style="text-align: left;">算子类型</td>
<td style="text-align: left;">Add</td>
</tr>
<tr>
<td style="text-align: left;">输入 x</td>
<td style="text-align: left;">shape (8,2048), float, ND</td>
</tr>
<tr>
<td style="text-align: left;">输入 y</td>
<td style="text-align: left;">shape (8,2048), float, ND</td>
</tr>
<tr>
<td style="text-align: left;">输出 z</td>
<td style="text-align: left;">shape (8,2048), float, ND</td>
</tr>
<tr>
<td style="text-align: left;">核函数名</td>
<td style="text-align: left;">add</td>
</tr>
<tr>
<td style="text-align: left;">使用核数</td>
<td style="text-align: left;">8</td>
</tr>
</table>

### 多核并行策略

数据整体长度为 8 × 2048 = 16384 个元素，平均分配到 8 个核上运行，每个核处理 2048 个元素。单核内再进行数据切块（Tiling），实现流水并行。

**输入/输出说明**：
- **输入向量 x**：shape (8, 2048)，float 类型，全部填充为 `valueX = 1.2f`，即 `x = [1.2, 1.2, ..., 1.2]`（共 16384 个 1.2）。
- **输入向量 y**：shape (8, 2048)，float 类型，全部填充为 `valueY = 2.3f`，即 `y = [2.3, 2.3, ..., 2.3]`（共 16384 个 2.3）。
- **输出向量 z**：shape (8, 2048)，float 类型，z = x + y，每个元素为 `1.2 + 2.3 = 3.5`，即 `z = [3.5, 3.5, ..., 3.5]`。
- **Golden（参考值）**：CPU 计算的 `valueX + valueY = 3.5`，用于与 NPU 算子输出逐元素比对验证。

**多核切分对应关系**：
<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">核号</th>
<th style="text-align: left;">数据范围</th>
<th style="text-align: left;">输入 x</th>
<th style="text-align: left;">输入 y</th>
<th style="text-align: left;">输出 z</th>
</tr>
<tr>
<td style="text-align: left;">核 0</td>
<td style="text-align: left;">[0, 2048)</td>
<td style="text-align: left;">[1.2, ...]</td>
<td style="text-align: left;">[2.3, ...]</td>
<td style="text-align: left;">[3.5, ...]</td>
</tr>
<tr>
<td style="text-align: left;">核 1</td>
<td style="text-align: left;">[2048, 4096)</td>
<td style="text-align: left;">[1.2, ...]</td>
<td style="text-align: left;">[2.3, ...]</td>
<td style="text-align: left;">[3.5, ...]</td>
</tr>
<tr>
<td style="text-align: left;">...</td>
<td style="text-align: left;">...</td>
<td style="text-align: left;">...</td>
<td style="text-align: left;">...</td>
<td style="text-align: left;">...</td>
</tr>
<tr>
<td style="text-align: left;">核 7</td>
<td style="text-align: left;">[14336, 16384)</td>
<td style="text-align: left;">[1.2, ...]</td>
<td style="text-align: left;">[2.3, ...]</td>
<td style="text-align: left;">[3.5, ...]</td>
</tr>
</table>

> 每个核独立处理自己负责的 2048 个元素，通过 `GetBlockIdx()` 获取核号并计算数据偏移量。8 个核并行执行，最终拼接的结果与串行计算完全一致。


## 核函数开发

### 1.1 夀文件引入 & 定义 BufferNum

进行算子开发时，首先要在源文件中导入必要的头文件。`BUFFER_NUM = 2` 用于开启 Double Buffer，实现 CopyIn/Compute/CopyOut 三级流水并行。

In [ ]:
%%writefile Sources/add.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

constexpr uint32_t BUFFER_NUM = 2; // tensor num for each queue


### 1.2 tiling 结构体创建

`AddCustomTilingData` 结构体保存数据切分参数：

- `totalLength`：待处理的数据总大小（8 * 2048）
- `tileNum`：每个核需要计算的数据块个数

In [ ]:
%%writefile -a Sources/add.asc

struct AddCustomTilingData
{
    uint32_t totalLength;
    uint32_t tileNum;
};


### 1.3 创建核函数类

矢量编程范式把算子的实现流程分为 3 个基本任务：**CopyIn**（搬入）、**Compute**（计算）、**CopyOut**（搬出）。

- **CopyIn**：将输入数据从 Global Memory 搬运到 Local Memory，完成搬运后执行入队列操作；
- **Compute**：完成队列出队后，从 Local Memory 获取数据并计算，计算完成后执行入队操作；
- **CopyOut**：完成队列出队后，将计算结果从 Local Memory 搬运到 Global Memory。

In [ ]:
%%writefile -a Sources/add.asc

class KernelAdd {
public:
    __aicore__ inline KernelAdd(){}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum);
    __aicore__ inline void Process();

private:
    __aicore__ inline void CopyIn(int32_t progress);
    __aicore__ inline void Compute(int32_t progress);
    __aicore__ inline void CopyOut(int32_t progress);

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> yGm;
    AscendC::GlobalTensor<float> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};


### 1.4 Init 函数实现

初始化函数 Init 主要完成：
- 设置输入输出 Global Tensor 的 Global Memory 内存地址（通过 `GetBlockIdx()` 实现多核数据切分）
- 通过 TPipe 内存管理对象为输入输出 Queue 分配内存

每个核处理 `blockLength = totalLength / GetBlockNum()` 个元素，单核内再切分成 `tileNum * BUFFER_NUM` 块。

In [ ]:
%%writefile -a Sources/add.asc

__aicore__ inline void KernelAdd::Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
{
     this->blockLength = totalLength / AscendC::GetBlockNum();
     this->tileNum = tileNum;
     this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
     xGm.SetGlobalBuffer((__gm__ float *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     yGm.SetGlobalBuffer((__gm__ float *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     zGm.SetGlobalBuffer((__gm__ float *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(float));
}


### 1.5 Process / CopyIn / Compute / CopyOut 函数实现

Process 函数中循环调用三个流水任务，循环次数为 `tileNum * BUFFER_NUM`。

In [ ]:
%%writefile -a Sources/add.asc

__aicore__ inline void KernelAdd::Process()
{
    int32_t loopCount = this->tileNum * BUFFER_NUM;
    for (int32_t i = 0; i < loopCount; i++) {
        CopyIn(i);
        Compute(i);
        CopyOut(i);
    }
}

__aicore__ inline void KernelAdd::CopyIn(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.AllocTensor<float>();
    AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
    AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);
}

__aicore__ inline void KernelAdd::Compute(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.DeQue<float>();
    AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
    AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
    outQueueZ.EnQue<float>(zLocal);
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
}

__aicore__ inline void KernelAdd::CopyOut(int32_t progress)
{
    AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
    AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
    outQueueZ.FreeTensor(zLocal);
}


### 1.6 核函数定义

核函数是 Ascend C 算子设备侧实现的入口，使用 `__global__ __aicore__` 限定符标识，在核函数中调用算子类的 Init 和 Process 函数。

In [ ]:
%%writefile -a Sources/add.asc

__global__ __aicore__ void add(GM_ADDR x, GM_ADDR y, GM_ADDR z, AddCustomTilingData tiling)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    KernelAdd op;
    op.Init(x, y, z, tiling.totalLength, tiling.tileNum);
    op.Process();
}


## 算子调用（Host 侧）

### 2.1 核函数调用

通过内核调用符 `<<<blockDim, nullptr, stream>>>` 调用核函数，`blockDim = 8` 表示在 8 个核上并行执行。整个流程包括：初始化 → 申请 Host/Device 内存 → 数据拷贝 → 调用核函数 → 同步 → 结果拷回 → 释放资源。

In [ ]:
%%writefile -a Sources/add.asc

std::vector<float> kernel_add(std::vector<float> &x, std::vector<float> &y)
{
    constexpr uint32_t blockDim = 8;
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    AddCustomTilingData tiling = {/*totalLength:*/totalLength, /*tileNum:*/8};
    uint8_t *xHost = reinterpret_cast<uint8_t *>(x.data());
    uint8_t *yHost = reinterpret_cast<uint8_t *>(y.data());
    uint8_t *zHost = nullptr;
    uint8_t *xDevice = nullptr;
    uint8_t *yDevice = nullptr;
    uint8_t *zDevice = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);
    aclrtMallocHost((void **)(&zHost), totalByteSize);
    aclrtMalloc((void **)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, yHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    add<<<blockDim, nullptr, stream>>>(xDevice, yDevice, zDevice, tiling);
    aclrtSynchronizeStream(stream);
    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> z((float *)zHost, (float *)(zHost + totalLength));
    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return z;
}


### 2.2 计算结果比对 & 验证主程序

`VerifyResult` 比对实际输出与 golden 值；`main` 函数生成输入数据并调用算子。

In [ ]:
%%writefile -a Sources/add.asc

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (std::equal(output.begin(), output.end(), golden.begin())) {
        std::cout << "[Success] 精度验证通过！" << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] 精度验证失败！" << std::endl;
        return 1;
    }
}

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 1.2f;
    constexpr float valueY = 2.3f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);

    std::vector<float> output = kernel_add(x, y);

    std::vector<float> golden(totalLength, valueX + valueY);
    return VerifyResult(output, golden);
}


## 编译运行

**编译命令说明**：`bisheng` 是昇腾专用的 C/C++ 编译器，支持 Ascend C 扩展语法。`--npu-arch=dav-2201` 指定目标 NPU 架构（对应昇腾 910B 系列），`-o add` 指定输出可执行文件名。

**运行预期结果**：
```text
Output: 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5...
Golden: 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5 3.5...
[Success] 精度验证通过！
```

**输入→输出对应关系**：
- 输入 x = [1.2, 1.2, ..., 1.2]（16384 个 1.2）
- 输入 y = [2.3, 2.3, ..., 2.3]（16384 个 2.3）
- 算子执行加法：z[i] = x[i] + y[i] = 1.2 + 2.3 = 3.5
- 输出 z = [3.5, 3.5, ..., 3.5]（16384 个 3.5）
- Output 与 Golden 完全一致，精度验证通过

> **为什么全部是 3.5？** 因为输入向量 x 和 y 分别用常量 1.2 和 2.3 填充，加法算子对每个元素独立计算 `z[i] = x[i] + y[i]`，结果必然全部相同。这种常量填充的测试数据便于验证——如果输出中出现了非 3.5 的值，说明算子实现有误。


In [ ]:
!bisheng Sources/add.asc --npu-arch=dav-2201 -o add

In [ ]:
!./add

## 阶段一总结

- ✅ 成功实现双向量加法算子
- ✅ 理解 CopyIn-Compute-CopyOut 三级流水
- ✅ 掌握 8 核并行的数据切分方式


---

# 阶段二：双向量加法（32核）—— 多核并行扩展

## 学习目标

- 理解核数变化对并行策略的影响
- 掌握 blockDim 和 GetBlockNum() 的对应关系
- 观察不同核数下的性能表现

## 与阶段一的区别

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">阶段一</th>
<th style="text-align: left;">阶段二</th>
</tr>
<tr>
<td style="text-align: left;">输入 shape</td>
<td style="text-align: left;">(8, 2048)</td>
<td style="text-align: left;">(32, 2048)</td>
</tr>
<tr>
<td style="text-align: left;">数据总量</td>
<td style="text-align: left;">16,384 元素</td>
<td style="text-align: left;">65,536 元素</td>
</tr>
<tr>
<td style="text-align: left;">使用核数</td>
<td style="text-align: left;">8</td>
<td style="text-align: left;">32</td>
</tr>
<tr>
<td style="text-align: left;">每核处理</td>
<td style="text-align: left;">2,048 元素</td>
<td style="text-align: left;">2,048 元素</td>
</tr>
</table>

**关键变化**：数据总量增至 4 倍，核数增至 4 倍，每核处理数据量保持不变。

**输入/输出说明**：
- **输入向量 x**：shape (32, 2048)，全部填充为 `valueX = 2.2f`，即 65536 个 2.2（注意与阶段一的 1.2 区分）。
- **输入向量 y**：shape (32, 2048)，全部填充为 `valueY = 2.3f`，即 65536 个 2.3。
- **输出向量 z**：z = x + y，每个元素为 `2.2 + 2.3 = 4.5`，即 65536 个 4.5。
- **Golden**：`valueX + valueY = 4.5`，用于逐元素比对验证。

**多核切分对应关系**：
<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">核号</th>
<th style="text-align: left;">数据范围</th>
<th style="text-align: left;">输入 x</th>
<th style="text-align: left;">输入 y</th>
<th style="text-align: left;">输出 z</th>
</tr>
<tr>
<td style="text-align: left;">核 0</td>
<td style="text-align: left;">[0, 2048)</td>
<td style="text-align: left;">[2.2, ...]</td>
<td style="text-align: left;">[2.3, ...]</td>
<td style="text-align: left;">[4.5, ...]</td>
</tr>
<tr>
<td style="text-align: left;">核 1</td>
<td style="text-align: left;">[2048, 4096)</td>
<td style="text-align: left;">[2.2, ...]</td>
<td style="text-align: left;">[2.3, ...]</td>
<td style="text-align: left;">[4.5, ...]</td>
</tr>
<tr>
<td style="text-align: left;">...</td>
<td style="text-align: left;">...</td>
<td style="text-align: left;">...</td>
<td style="text-align: left;">...</td>
<td style="text-align: left;">...</td>
</tr>
<tr>
<td style="text-align: left;">核 31</td>
<td style="text-align: left;">[63488, 65536)</td>
<td style="text-align: left;">[2.2, ...]</td>
<td style="text-align: left;">[2.3, ...]</td>
<td style="text-align: left;">[4.5, ...]</td>
</tr>
</table>

> **与阶段一的对比**：阶段一用 8 核处理 16384 个元素，阶段二用 32 核处理 65536 个元素。每核处理量不变（2048），但总数据量和并行度都提升了 4 倍。理论上阶段二的吞吐量是阶段一的 4 倍（前提是硬件有足够的 AI Core）。

### 思考题

1. 为什么 `GetBlockNum()` 的返回值必须等于 `blockDim`？
2. 如果数据总量不能被核数整除，应该如何处理？


## 核函数开发

代码与阶段一基本相同，主要修改 `blockDim` 和 `totalLength`。以下逐步写入 `Sources/add32.asc`。

In [ ]:
%%writefile Sources/add32.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

constexpr uint32_t BUFFER_NUM = 2; // tensor num for each queue

struct AddCustomTilingData
{
    uint32_t totalLength;
    uint32_t tileNum;
};

class KernelAdd {
public:
    __aicore__ inline KernelAdd(){}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum);
    __aicore__ inline void Process();

private:
    __aicore__ inline void CopyIn(int32_t progress);
    __aicore__ inline void Compute(int32_t progress);
    __aicore__ inline void CopyOut(int32_t progress);

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> yGm;
    AscendC::GlobalTensor<float> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

__aicore__ inline void KernelAdd::Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
{
     this->blockLength = totalLength / AscendC::GetBlockNum();
     this->tileNum = tileNum;
     this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
     xGm.SetGlobalBuffer((__gm__ float *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     yGm.SetGlobalBuffer((__gm__ float *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     zGm.SetGlobalBuffer((__gm__ float *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(float));
}

__aicore__ inline void KernelAdd::Process()
{
    int32_t loopCount = this->tileNum * BUFFER_NUM;
    for (int32_t i = 0; i < loopCount; i++) {
        CopyIn(i);
        Compute(i);
        CopyOut(i);
    }
}

__aicore__ inline void KernelAdd::CopyIn(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.AllocTensor<float>();
    AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
    AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);
}

__aicore__ inline void KernelAdd::Compute(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.DeQue<float>();
    AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
    AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
    outQueueZ.EnQue<float>(zLocal);
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
}

__aicore__ inline void KernelAdd::CopyOut(int32_t progress)
{
    AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
    AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
    outQueueZ.FreeTensor(zLocal);
}

__global__ __aicore__ void add(GM_ADDR x, GM_ADDR y, GM_ADDR z, AddCustomTilingData tiling)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    KernelAdd op;
    op.Init(x, y, z, tiling.totalLength, tiling.tileNum);
    op.Process();
}


## 算子调用（Host 侧）

**关键差异对比：**

```cpp
// 阶段一：8 核，数据总量 8×2048
constexpr uint32_t blockDim = 8;
constexpr uint32_t totalLength = 8 * 2048;
constexpr float valueX = 1.2f;

// 阶段二：32 核，数据总量 32×2048
constexpr uint32_t blockDim = 32;
constexpr uint32_t totalLength = 32 * 2048;
constexpr float valueX = 2.2f;  // 与阶段一区分
```

In [ ]:
%%writefile -a Sources/add32.asc

std::vector<float> kernel_add(std::vector<float> &x, std::vector<float> &y)
{
    constexpr uint32_t blockDim = 32;  // 32 核并行
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    AddCustomTilingData tiling = {/*totalLength:*/totalLength, /*tileNum:*/8};
    uint8_t *xHost = reinterpret_cast<uint8_t *>(x.data());
    uint8_t *yHost = reinterpret_cast<uint8_t *>(y.data());
    uint8_t *zHost = nullptr;
    uint8_t *xDevice = nullptr;
    uint8_t *yDevice = nullptr;
    uint8_t *zDevice = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);
    aclrtMallocHost((void **)(&zHost), totalByteSize);
    aclrtMalloc((void **)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, yHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    add<<<blockDim, nullptr, stream>>>(xDevice, yDevice, zDevice, tiling);
    aclrtSynchronizeStream(stream);
    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> z((float *)zHost, (float *)(zHost + totalLength));
    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return z;
}

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (std::equal(output.begin(), output.end(), golden.begin())) {
        std::cout << "[Success] 精度验证通过！" << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] 精度验证失败！" << std::endl;
        return 1;
    }
}

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 32 * 2048;  // 数据总量改为 32×2048
    constexpr float valueX = 2.2f;               // 与阶段一区分
    constexpr float valueY = 2.3f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);

    std::vector<float> output = kernel_add(x, y);

    std::vector<float> golden(totalLength, valueX + valueY);
    return VerifyResult(output, golden);
}


## 编译运行

**运行预期结果**：
```text
Output: 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5...
Golden: 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5...
[Success] 精度验证通过！
```

**输入→输出对应关系**：
- 输入 x = [2.2, 2.2, ..., 2.2]（65536 个 2.2）
- 输入 y = [2.3, 2.3, ..., 2.3]（65536 个 2.3）
- 算子执行加法：z[i] = x[i] + y[i] = 2.2 + 2.3 = 4.5
- 输出 z = [4.5, 4.5, ..., 4.5]（65536 个 4.5）

> **与阶段一结果的对比**：阶段一输出全为 3.5（1.2+2.3），阶段二输出全为 4.5（2.2+2.3）。输入常量不同导致输出不同，但算子逻辑（加法）相同。32 个核并行处理，每个核仍处理 2048 个元素，验证了核数扩展时数据切分的正确性。


In [ ]:
!bisheng Sources/add32.asc --npu-arch=dav-2201 -o add32

In [ ]:
!./add32

## 阶段二总结

- ✅ 理解核数扩展时的数据切分策略
- ✅ 掌握多核并行的性能影响因素
- ✅ 学会分析不同核数配置的适用场景


---

# 阶段三：三向量加法 —— 多输入算子扩展

## 学习目标

- 掌握多输入算子的实现方法
- 理解队列资源和内存管理的扩展
- 学习多输入场景下的流水线调度

## 与阶段一的区别

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">阶段一</th>
<th style="text-align: left;">阶段三</th>
</tr>
<tr>
<td style="text-align: left;">输入数量</td>
<td style="text-align: left;">2 (x, y)</td>
<td style="text-align: left;">3 (x, y, w)</td>
</tr>
<tr>
<td style="text-align: left;">计算公式</td>
<td style="text-align: left;">z = x + y</td>
<td style="text-align: left;">z = x + y + w</td>
</tr>
<tr>
<td style="text-align: left;">输入队列</td>
<td style="text-align: left;">inQueueX, inQueueY</td>
<td style="text-align: left;">inQueueX, inQueueY, inQueueW</td>
</tr>
</table>

**输入/输出说明**：
- **输入向量 x**：shape (8, 2048)，全部填充为 `valueX = 1.2f`，即 16384 个 1.2。
- **输入向量 y**：shape (8, 2048)，全部填充为 `valueY = 2.3f`，即 16384 个 2.3。
- **输入向量 w**：shape (8, 2048)，全部填充为 `valueW = 3.4f`，即 16384 个 3.4。
- **输出向量 z**：z = x + y + w，每个元素为 `1.2 + 2.3 + 3.4 = 6.9`，即 16384 个 6.9。
- **Golden**：`valueX + valueY + valueW = 6.9`，用于逐元素比对验证。

**Compute 函数中的两次 Add**：
- 第 1 次：`Add(zLocal, xLocal, yLocal, tileLength)` → z = x + y = [3.5, 3.5, ...]
- 第 2 次：`Add(zLocal, zLocal, wLocal, tileLength)` → z = z + w = [6.9, 6.9, ...]

> **为什么需要两次 Add？** Ascend C 的 `Add` 接口只支持两个输入向量的加法，不支持三个。要实现 `z = x + y + w`，需要分两步：先计算 `z = x + y`，再将结果与 w 相加 `z = z + w`。这是多输入算子的常见处理方式——将 N 个输入的运算分解为 N-1 次双输入运算。

### 思考题

1. 三向量相加能否用一次 Add 接口完成？为什么？
2. 增加输入数量对 Double Buffer 机制有什么影响？
3. 如果有 4 个输入向量相加，代码应该如何扩展？


## 核函数开发

与阶段一相比，主要修改 8 处：增加第三个输入队列 `inQueueW`、GlobalTensor `wGm`、Init/CopyIn/Compute 函数参数、核函数参数、Host 侧内存操作。以下写入 `Sources/add3.asc`。

In [ ]:
%%writefile Sources/add3.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

constexpr uint32_t BUFFER_NUM = 2; // tensor num for each queue

struct AddCustomTilingData
{
    uint32_t totalLength;
    uint32_t tileNum;
};

class KernelAdd {
public:
    __aicore__ inline KernelAdd(){}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR w, GM_ADDR z, uint32_t totalLength, uint32_t tileNum);
    __aicore__ inline void Process();

private:
    __aicore__ inline void CopyIn(int32_t progress);
    __aicore__ inline void Compute(int32_t progress);
    __aicore__ inline void CopyOut(int32_t progress);

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY, inQueueW;  // 增加第三个输入队列
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> yGm;
    AscendC::GlobalTensor<float> wGm;  // 增加第三个 GlobalTensor
    AscendC::GlobalTensor<float> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

__aicore__ inline void KernelAdd::Init(GM_ADDR x, GM_ADDR y, GM_ADDR w, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
{
     this->blockLength = totalLength / AscendC::GetBlockNum();
     this->tileNum = tileNum;
     this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
     xGm.SetGlobalBuffer((__gm__ float *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     yGm.SetGlobalBuffer((__gm__ float *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     wGm.SetGlobalBuffer((__gm__ float *)w + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     zGm.SetGlobalBuffer((__gm__ float *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(inQueueW, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(float));
}

__aicore__ inline void KernelAdd::Process()
{
    int32_t loopCount = this->tileNum * BUFFER_NUM;
    for (int32_t i = 0; i < loopCount; i++) {
        CopyIn(i);
        Compute(i);
        CopyOut(i);
    }
}

__aicore__ inline void KernelAdd::CopyIn(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.AllocTensor<float>();
    AscendC::LocalTensor<float> wLocal = inQueueW.AllocTensor<float>();
    AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
    AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
    AscendC::DataCopy(wLocal, wGm[progress * this->tileLength], this->tileLength);
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);
    inQueueW.EnQue(wLocal);
}

__aicore__ inline void KernelAdd::Compute(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.DeQue<float>();
    AscendC::LocalTensor<float> wLocal = inQueueW.DeQue<float>();
    AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
    // 核心变化：分两次加法完成三向量相加
    AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);   // z = x + y
    AscendC::Add(zLocal, zLocal, wLocal, this->tileLength);   // z = z + w
    outQueueZ.EnQue<float>(zLocal);
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
    inQueueW.FreeTensor(wLocal);
}

__aicore__ inline void KernelAdd::CopyOut(int32_t progress)
{
    AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
    AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
    outQueueZ.FreeTensor(zLocal);
}

__global__ __aicore__ void add(GM_ADDR x, GM_ADDR y, GM_ADDR w, GM_ADDR z, AddCustomTilingData tiling)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    KernelAdd op;
    op.Init(x, y, w, z, tiling.totalLength, tiling.tileNum);
    op.Process();
}


## 算子调用（Host 侧）

Host 侧需要增加 `w` 相关的内存申请、拷贝、释放以及 `<<<...>>>` 调用参数。

In [ ]:
%%writefile -a Sources/add3.asc

std::vector<float> kernel_add(std::vector<float> &x, std::vector<float> &y, std::vector<float> &w)
{
    constexpr uint32_t blockDim = 8;
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    AddCustomTilingData tiling = {/*totalLength:*/totalLength, /*tileNum:*/8};
    uint8_t *xHost = reinterpret_cast<uint8_t *>(x.data());
    uint8_t *yHost = reinterpret_cast<uint8_t *>(y.data());
    uint8_t *wHost = reinterpret_cast<uint8_t *>(w.data());
    uint8_t *zHost = nullptr;
    uint8_t *xDevice = nullptr;
    uint8_t *yDevice = nullptr;
    uint8_t *wDevice = nullptr;
    uint8_t *zDevice = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);
    aclrtMallocHost((void **)(&zHost), totalByteSize);
    aclrtMalloc((void **)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&wDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, yHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(wDevice, totalByteSize, wHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    add<<<blockDim, nullptr, stream>>>(xDevice, yDevice, wDevice, zDevice, tiling);
    aclrtSynchronizeStream(stream);
    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> z((float *)zHost, (float *)(zHost + totalLength));
    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(wDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return z;
}

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (std::equal(output.begin(), output.end(), golden.begin())) {
        std::cout << "[Success] 精度验证通过！" << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] 精度验证失败！" << std::endl;
        return 1;
    }
}

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 1.2f;
    constexpr float valueY = 2.3f;
    constexpr float valueW = 3.4f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);
    std::vector<float> w(totalLength, valueW);

    std::vector<float> output = kernel_add(x, y, w);

    std::vector<float> golden(totalLength, valueX + valueY + valueW);
    return VerifyResult(output, golden);
}


## 编译运行

**运行预期结果**：
```text
Output: 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9...
Golden: 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9 6.9...
[Success] 精度验证通过！
```

**输入→输出对应关系**：
- 输入 x = [1.2, 1.2, ..., 1.2]（16384 个 1.2）
- 输入 y = [2.3, 2.3, ..., 2.3]（16384 个 2.3）
- 输入 w = [3.4, 3.4, ..., 3.4]（16384 个 3.4）
- 算子分两步执行：第 1 步 z = x + y = [3.5, ...]，第 2 步 z = z + w = [6.9, ...]
- 输出 z = [6.9, 6.9, ..., 6.9]（16384 个 6.9）

> **为什么是 6.9？** 三个输入常量之和：1.2 + 2.3 + 3.4 = 6.9。Compute 函数中调用了两次 `Add` 指令：第一次将 x 和 y 相加得到中间结果 3.5，第二次将中间结果与 w 相加得到最终结果 6.9。验证通过说明两次加法的串联执行正确，多输入算子的数据搬运和队列管理无误。


In [ ]:
!bisheng Sources/add3.asc --npu-arch=dav-2201 -o add3

In [ ]:
!./add3

## 阶段三总结

- ✅ 理解多输入算子的实现方法
- ✅ 掌握队列资源和内存管理的扩展方式
- ✅ 学会在多输入场景下维护流水线并行


---

# 阶段四：双向量乘法 —— 计算指令变化

## 学习目标

- 理解不同计算指令（Add vs Mul）的使用方法
- 掌握算子的快速移植和修改技巧
- 对比加法与乘法的精度验证方式

## 与阶段一的区别

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">阶段一</th>
<th style="text-align: left;">阶段四</th>
</tr>
<tr>
<td style="text-align: left;">计算类型</td>
<td style="text-align: left;">加法</td>
<td style="text-align: left;">乘法</td>
</tr>
<tr>
<td style="text-align: left;">Ascend C 接口</td>
<td style="text-align: left;"><code>AscendC::Add()</code></td>
<td style="text-align: left;"><code>AscendC::Mul()</code></td>
</tr>
<tr>
<td style="text-align: left;">Golden 计算</td>
<td style="text-align: left;"><code>valueX + valueY</code></td>
<td style="text-align: left;"><code>valueX * valueY</code></td>
</tr>
</table>

**输入/输出说明**：
- **输入向量 x**：shape (8, 2048)，全部填充为 `valueX = 1.2f`，即 16384 个 1.2。
- **输入向量 y**：shape (8, 2048)，全部填充为 `valueY = 2.3f`，即 16384 个 2.3。
- **输出向量 z**：z = x * y（逐元素乘法），每个元素为 `1.2 × 2.3 = 2.76`，即 16384 个 2.76。
- **Golden**：`valueX * valueY = 2.76`，用于逐元素比对验证。

> **与阶段一的关键差异**：阶段一用 `AscendC::Add` 指令做加法，输出 3.5（1.2+2.3）；阶段四用 `AscendC::Mul` 指令做乘法，输出 2.76（1.2×2.3）。输入数据完全相同，仅计算指令不同，输出结果因此不同。这验证了"相同输入+不同算子=不同输出"的对应关系，帮助学生理解算子选择对结果的决定性影响。

### 思考题

1. Ascend C 还支持哪些矢量计算指令？（如 Sub, Div, Max, Min 等）
2. 如果要实现 `z = x + y * w`，应该如何组合指令？
3. 整数类型和浮点类型的指令有什么区别？


## 核函数开发

与阶段一相比，仅需修改两处：Compute 函数中 `Add → Mul`，main 函数中 golden 计算 `+ → *`。以下写入 `Sources/mul.asc`。

In [ ]:
%%writefile Sources/mul.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

constexpr uint32_t BUFFER_NUM = 2; // tensor num for each queue

struct AddCustomTilingData
{
    uint32_t totalLength;
    uint32_t tileNum;
};

class KernelAdd {
public:
    __aicore__ inline KernelAdd(){}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum);
    __aicore__ inline void Process();

private:
    __aicore__ inline void CopyIn(int32_t progress);
    __aicore__ inline void Compute(int32_t progress);
    __aicore__ inline void CopyOut(int32_t progress);

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> yGm;
    AscendC::GlobalTensor<float> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

__aicore__ inline void KernelAdd::Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
{
     this->blockLength = totalLength / AscendC::GetBlockNum();
     this->tileNum = tileNum;
     this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
     xGm.SetGlobalBuffer((__gm__ float *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     yGm.SetGlobalBuffer((__gm__ float *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     zGm.SetGlobalBuffer((__gm__ float *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(float));
}

__aicore__ inline void KernelAdd::Process()
{
    int32_t loopCount = this->tileNum * BUFFER_NUM;
    for (int32_t i = 0; i < loopCount; i++) {
        CopyIn(i);
        Compute(i);
        CopyOut(i);
    }
}

__aicore__ inline void KernelAdd::CopyIn(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.AllocTensor<float>();
    AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
    AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
    inQueueX.EnQue(xLocal);
    inQueueY.EnQue(yLocal);
}

__aicore__ inline void KernelAdd::Compute(int32_t progress)
{
    AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.DeQue<float>();
    AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
    // 关键变化：Add → Mul
    AscendC::Mul(zLocal, xLocal, yLocal, this->tileLength);
    outQueueZ.EnQue<float>(zLocal);
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
}

__aicore__ inline void KernelAdd::CopyOut(int32_t progress)
{
    AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
    AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
    outQueueZ.FreeTensor(zLocal);
}

__global__ __aicore__ void add(GM_ADDR x, GM_ADDR y, GM_ADDR z, AddCustomTilingData tiling)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    KernelAdd op;
    op.Init(x, y, z, tiling.totalLength, tiling.tileNum);
    op.Process();
}


## 算子调用（Host 侧）

Host 侧代码与阶段一基本一致，仅 golden 计算从 `valueX + valueY` 改为 `valueX * valueY`。

In [ ]:
%%writefile -a Sources/mul.asc

std::vector<float> kernel_add(std::vector<float> &x, std::vector<float> &y)
{
    constexpr uint32_t blockDim = 8;
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    AddCustomTilingData tiling = {/*totalLength:*/totalLength, /*tileNum:*/8};
    uint8_t *xHost = reinterpret_cast<uint8_t *>(x.data());
    uint8_t *yHost = reinterpret_cast<uint8_t *>(y.data());
    uint8_t *zHost = nullptr;
    uint8_t *xDevice = nullptr;
    uint8_t *yDevice = nullptr;
    uint8_t *zDevice = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);
    aclrtMallocHost((void **)(&zHost), totalByteSize);
    aclrtMalloc((void **)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, yHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    add<<<blockDim, nullptr, stream>>>(xDevice, yDevice, zDevice, tiling);
    aclrtSynchronizeStream(stream);
    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> z((float *)zHost, (float *)(zHost + totalLength));
    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
    return z;
}

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (std::equal(output.begin(), output.end(), golden.begin())) {
        std::cout << "[Success] 精度验证通过！" << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] 精度验证失败！" << std::endl;
        return 1;
    }
}

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 1.2f;
    constexpr float valueY = 2.3f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);

    std::vector<float> output = kernel_add(x, y);

    // 关键变化：golden 计算从 + 改为 *
    std::vector<float> golden(totalLength, valueX * valueY);
    return VerifyResult(output, golden);
}


## 编译运行

**运行预期结果**：
```text
Output: 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76...
Golden: 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76 2.76...
[Success] 精度验证通过！
```

**输入→输出对应关系**：
- 输入 x = [1.2, 1.2, ..., 1.2]（16384 个 1.2）
- 输入 y = [2.3, 2.3, ..., 2.3]（16384 个 2.3）
- 算子执行乘法：z[i] = x[i] × y[i] = 1.2 × 2.3 = 2.76
- 输出 z = [2.76, 2.76, ..., 2.76]（16384 个 2.76）

> **四阶段结果对比汇总**：
> | 阶段 | 输入 | 算子 | 输出 |
> |------|------|------|------|
> | 阶段一 | x=1.2, y=2.3 | z=x+y | 3.5 |
> | 阶段二 | x=2.2, y=2.3 | z=x+y | 4.5 |
> | 阶段三 | x=1.2, y=2.3, w=3.4 | z=x+y+w | 6.9 |
> | 阶段四 | x=1.2, y=2.3 | z=x*y | 2.76 |
>
> 每个阶段的输入常量和算子类型都不同，因此输出结果也不同。通过对比四个阶段的输入输出，可以清晰地理解"输入数据 + 算子类型 → 输出结果"的对应关系。


In [ ]:
!bisheng Sources/mul.asc --npu-arch=dav-2201 -o mul

In [ ]:
!./mul

## 阶段四总结

- ✅ 掌握 Ascend C 计算指令的替换方法
- ✅ 理解算子移植的最小修改原则
- ✅ 学会不同计算类型的精度验证方法


---

# 总结与进阶

## 四个阶段学习路线图

```
阶段一（双向量加法，8核）
    ↓ 掌握：基本流程 + 三级流水
阶段二（双向量加法，32核）
    ↓ 掌握：多核并行 + 数据切分
阶段三（三向量加法）
    ↓ 掌握：多输入 + 资源管理
阶段四（双向量乘法）
    ↓ 掌握：计算指令变化 + 快速移植
```

## 核心知识点汇总

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">知识点</th>
<th style="text-align: left;">阶段一</th>
<th style="text-align: left;">阶段二</th>
<th style="text-align: left;">阶段三</th>
<th style="text-align: left;">阶段四</th>
</tr>
<tr>
<td style="text-align: left;">核函数定义</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
</tr>
<tr>
<td style="text-align: left;">算子类设计</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
</tr>
<tr>
<td style="text-align: left;">三级流水</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
</tr>
<tr>
<td style="text-align: left;">Double Buffer</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;">✓</td>
</tr>
<tr>
<td style="text-align: left;">核数配置</td>
<td style="text-align: left;">✓ (8核)</td>
<td style="text-align: left;">✓ (32核)</td>
<td style="text-align: left;">✓ (8核)</td>
<td style="text-align: left;">✓ (8核)</td>
</tr>
<tr>
<td style="text-align: left;">多输入支持</td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;">✓</td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;">计算指令变化</td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;">✓</td>
</tr>
</table>

**知识点掌握情况解读**：
- **核函数定义、算子类设计、三级流水、Double Buffer** 是四个阶段都涉及的基础知识，学生在阶段一就应掌握。
- **核数配置**在阶段二中重点练习（8→32），学生需要理解 `blockDim` 与 `GetBlockNum()`/`GetBlockIdx()` 的关系。
- **多输入支持**是阶段三的专属内容，学生需要学会增加队列、GlobalTensor 和内存管理。
- **计算指令变化**是阶段四的专属内容，学生需要学会将 `Add` 替换为 `Mul` 等其他指令。

> **学习建议**：如果某个知识点在对应阶段未完全理解，建议回到该阶段重新研读代码并动手修改参数运行验证。四个阶段是递进关系，前面的基础不牢会影响后面的学习。

## 进阶练习建议

1. **组合算子**：实现 `z = x * y + w`，融合乘法和加法
2. **自定义计算**：实现 `z = x^2 + y^2`，使用 Mul 和 Add 组合
3. **性能对比**：对比不同核数、不同 tileNum 配置下的性能差异
4. **数据类型扩展**：将 float 类型扩展为 half 类型，比较精度和性能

## 常见问题 FAQ

**Q1: 为什么使用 8 个核而不是其他数量？**
A: 8 核是典型配置，便于教学演示。实际开发中应根据数据量和硬件规格选择最优核数。

**Q2: tileNum 如何选择？**
A: tileNum 影响内存利用率和流水并行效率。过小会导致流水线气泡，过大会增加调度开销。

**Q3: 如何调试算子？**
A: 可以使用 printf 打印调试信息，或使用 CANN 提供的 Profiling 工具进行性能分析。

**Q4: 代码中的 BUFFER_NUM = 2 可以改为其他值吗？**
A: 可以。BUFFER_NUM 决定 Double Buffer 的深度，增加深度可以隐藏更多访存延迟，但会增加内存占用。

## 附录：完整代码索引

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">文件</th>
<th style="text-align: left;">说明</th>
<th style="text-align: left;">关键参数</th>
</tr>
<tr>
<td style="text-align: left;"><code>Sources/add.asc</code></td>
<td style="text-align: left;">阶段一：双向量加法 8核</td>
<td style="text-align: left;">blockDim=8, totalLength=8×2048</td>
</tr>
<tr>
<td style="text-align: left;"><code>Sources/add32.asc</code></td>
<td style="text-align: left;">阶段二：双向量加法 32核</td>
<td style="text-align: left;">blockDim=32, totalLength=32×2048</td>
</tr>
<tr>
<td style="text-align: left;"><code>Sources/add3.asc</code></td>
<td style="text-align: left;">阶段三：三向量加法 8核</td>
<td style="text-align: left;">3个输入，2次Add指令</td>
</tr>
<tr>
<td style="text-align: left;"><code>Sources/mul.asc</code></td>
<td style="text-align: left;">阶段四：双向量乘法 8核</td>
<td style="text-align: left;">Mul指令，golden=valueX*valueY</td>
</tr>
</table>

---

### Notebook 设计特点

1. **循序渐进**：每个阶段只引入一个变化点，便于学生理解和消化
2. **对比教学**：通过表格和代码对比，直观展示各阶段的差异
3. **思考题设计**：每个阶段都配有思考题，引导学生主动思考
4. **总结归纳**：知识汇总表和进阶练习建议，帮助学生巩固和拓展


---

## 课后练习

请根据本实验内容完成以下题目进行自测。


**第1题**（单选题）Ascend C 算子开发的核函数使用什么限定符标识？

- A. __device__
- B. __global__ __aicore__
- C. __kernel__
- D. __npu__


In [ ]:
q1 = ''  # 填入你的选项，如 'B'
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')

**第2题**（单选题）三级流水线的正确顺序是？

- A. Compute → CopyIn → CopyOut
- B. CopyIn → Compute → CopyOut
- C. CopyOut → CopyIn → Compute
- D. CopyIn → CopyOut → Compute


In [ ]:
q2 = ''  # 填入你的选项，如 'B'
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')

**第3题**（单选题）BUFFER_NUM = 2 的作用是？

- A. 双缓冲，隐藏访存延迟
- B. 增加计算精度
- C. 减少内存使用
- D. 提高数据精度


In [ ]:
q3 = ''  # 填入你的选项，如 'B'
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')

**第4题**（单选题）阶段一中 blockDim 的值是多少？

- A. 4
- B. 8
- C. 16
- D. 32


In [ ]:
q4 = ''  # 填入你的选项，如 'B'
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')

**第5题**（单选题）阶段二相比阶段一的主要变化是？

- A. 核数 8→32
- B. 输入 2→3
- C. Add→Mul
- D. 数据类型变化


In [ ]:
q5 = ''  # 填入你的选项，如 'A'
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')

**第6题**（单选题）阶段三相比阶段一的主要变化是？

- A. 核数变化
- B. 输入 2→3，三向量加法
- C. 指令变化
- D. 数据类型变化


In [ ]:
q6 = ''  # 填入你的选项，如 'B'
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')

**第7题**（单选题）阶段四将 Add 指令替换为什么？

- A. Sub
- B. Mul
- C. Div
- D. Max


In [ ]:
q7 = ''  # 填入你的选项，如 'B'
print(f'第7题答案已记录：{q7}' if q7 else '请填入答案并运行本单元格')

**第8题**（单选题）Kernel 直调工程的特点是？

- A. 算子实现与调用代码在同一源文件
- B. 需要完整框架对接
- C. 只能在 CPU 运行
- D. 不支持多核


In [ ]:
q8 = ''  # 填入你的选项，如 'A'
print(f'第8题答案已记录：{q8}' if q8 else '请填入答案并运行本单元格')

**第9题**（单选题）编译 Ascend C 算子使用什么编译器？

- A. gcc
- B. bisheng
- C. clang
- D. msvc


In [ ]:
q9 = ''  # 填入你的选项，如 'B'
print(f'第9题答案已记录：{q9}' if q9 else '请填入答案并运行本单元格')

**第10题**（单选题）阶段一中输入 shape 固定为？

- A. (4, 2048)
- B. (8, 2048)
- C. (16, 2048)
- D. (32, 2048)


In [ ]:
q10 = ''  # 填入你的选项，如 'B'
print(f'第10题答案已记录：{q10}' if q10 else '请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / 'answer', Path.cwd().parent / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_03 import grade
grade(globals())

## 参考资料

- [昇腾社区 - Ascend C 算子开发](https://hiascend.com/document)
- [CANN 社区样例](https://gitee.com/ascend/samples)
